<a href="https://colab.research.google.com/github/AktanM11/AI-OI/blob/main/DAY1.1_WEEK5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install qdrant-client

In [ ]:
!pip install -U FlagEmbedding

In [ ]:
pip install langchain-groq

In [4]:
import json
import uuid
import os
from typing import List, Dict, Generator
from qdrant_client import QdrantClient
from google.colab import userdata
from FlagEmbedding import FlagAutoModel
from qdrant_client.models import Filter, FieldCondition, MatchValue
from typing import Literal
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END


In [ ]:
bge = FlagAutoModel.from_finetuned('BAAI/bge-m3',
                                      use_fp16=True)

In [6]:
qdrant_client = QdrantClient(url=userdata.get('QDRANT_URL'), api_key=userdata.get('QDRANT_API_KEY'))

In [7]:
collection_name = "policies_collection_bge"

In [8]:
def search_qdrant_bge(query_text: str, search_filter: Filter = None):
    # E5 model needs "query: " prefix for questions/searches
    query_dense = bge.encode(query_text)['dense_vecs']

    # Perform the search
    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_dense,
        query_filter=search_filter,
        limit=1
    )
    actual_points = results.points
    print(f"\n Results for query: '{query_text}'")
    if not results:
        print("No results found matching this filter.")
        return

    for idx, hit in enumerate(actual_points):
        print(f"\n[Result #{idx+1}]")
        print(f"-> Category: {hit.payload.get('category')}")
        print(f"-> Source File: {hit.payload.get('source_file')}")
        print(f"-> Text: {hit.payload.get('page_content')[:450]}...")

In [9]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    temperature=0,
    groq_api_key=userdata.get('GROQ_API_KEY'),
    model_name="llama-3.3-70b-versatile"
)

In [10]:
def retrieve_node(state: MessagesState):
    """
    Нода поиска. Работает с MessagesState, содержащим только последнее сообщение.
    """
    user_question = state["messages"][-1].content
    retrieved_chunks = []

    try:
        query_dense_bge = bge.encode(user_question)['dense_vecs']
        results_bge = qdrant_client.query_points(
            collection_name=collection_name,
            query=query_dense_bge,
            limit=1
        )
        if results_bge.points:
            chunk_text = results_bge.points[0].payload.get('page_content', '')
            retrieved_chunks.append(f"[Контекст из BGE-M3]: {chunk_text}")
    except Exception as e:
        print(f"Ошибка поиска BGE-M3: {e}")

    context_str = "\n\n".join(retrieved_chunks) if retrieved_chunks else "Контекст в базе данных отсутствует."

    return {
        "messages": [AIMessage(content=context_str, name="context_holder")]
    }


def generate_node(state: dict):
    """
    Нода генерации. Работает с любым стэйтом содержащим messages.
    """
    messages = state["messages"]
    context = messages[-1].content
    user_question = messages[-2].content


    system_prompt = (
        "Вы — корпоративный AI-ассистент поддержки сотрудников O!Store.\n"
        "Ваша задача — строго отвечать на вопросы пользователя на основе предоставленного контекста.\n"
        "Если в контексте нет четкого ответа на вопрос или модели поиска вернули ошибочные данные, "
        "вы ОБЯЗАНЫ строго ответить: 'К сожалению, я не могу ответить на этот вопрос на основе имеющихся регламентов O!Store.'\n"
        "Не придумывайте факты, не используйте внешние знания, которых нет в тексте.\n\n"
        f"ПРЕДОСТАВЛЕННЫЙ КОНТЕКСТ:\n{context}"
    )

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_question)
    ])

    return {
        "messages": [AIMessage(content=response.content)]
    }


workflow = StateGraph(MessagesState)

workflow.add_node("retriever", retrieve_node)
workflow.add_node("generator", generate_node)

workflow.add_edge(START, "retriever")
workflow.add_edge("retriever", "generator")
workflow.add_edge("generator", END)

app = workflow.compile()

In [11]:
def rewrite_query_node(state: MessagesState):
    """
    Переписывает запрос пользователя для улучшения поиска.
    """
    user_question = state["messages"][-1].content

    rewrite_prompt = (
        "Перепиши следующий вопрос так, чтобы он стал более точным и эффективным "
        "для семантического поиска в корпоративной базе знаний O!Store. "
        "Верни только переписанный вопрос, без пояснений.\n\n"
        f"Исходный вопрос: {user_question}"
    )

    rewritten = llm.invoke([HumanMessage(content=rewrite_prompt)])

    # Заменяем последнее сообщение переписанным запросом
    return {
        "messages": [HumanMessage(content=rewritten.content)]
    }


workflow_rewrite = StateGraph(MessagesState)
workflow_rewrite.add_node("rewrite_query", rewrite_query_node)
workflow_rewrite.add_node("retriever", retrieve_node)
workflow_rewrite.add_node("generator", generate_node)

workflow_rewrite.add_edge(START, "rewrite_query")
workflow_rewrite.add_edge("rewrite_query", "retriever")
workflow_rewrite.add_edge("retriever", "generator")
workflow_rewrite.add_edge("generator", END)

app_rewrite = workflow_rewrite.compile()

In [12]:
def generate_hypothesis(question: str) -> str:
    """Генерирует выдуманный но правдоподобный ответ для эмбеддинга"""
    prompt = (
        "Напиши короткий гипотетический ответ на вопрос (2-4 предложения). "
        "Пиши уверенно и конкретно, как будто знаешь ответ. "
        "Не говори что это гипотеза.\n\n"
        f"Вопрос: {question}"
    )
    # Здесь НЕТ ограничений — модель должна свободно выдумывать
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

def hyde_node(state: MessagesState):
    """
    HyDE: генерирует гипотетический ответ → эмбеддирует его → ищет похожие чанки.
    Вместо эмбеддинга вопроса — эмбеддинг придуманного ответа.
    """
    user_question = state["messages"][-1].content
    hyde_prompt = generate_hypothesis(user_question)

    retrieved_chunks = []

    try:
        hyp_dense = bge.encode(hyde_prompt)['dense_vecs']
        results = qdrant_client.query_points(
            collection_name=collection_name,
            query=hyp_dense,
            limit=3
        )
        for point in results.points:
            chunk_text = point.payload.get('page_content', '')
            retrieved_chunks.append(f"[HyDE контекст]: {chunk_text}")
    except Exception as e:
        print(f"Ошибка HyDE поиска: {e}")

    context_str = "\n\n".join(retrieved_chunks) if retrieved_chunks else "Контекст отсутствует."

    return {
        "messages": [AIMessage(content=context_str, name="context_holder")]
    }


workflow_hyde = StateGraph(MessagesState)
workflow_hyde.add_node("hyde_retriever", hyde_node)
workflow_hyde.add_node("generator", generate_node)

workflow_hyde.add_edge(START, "hyde_retriever")
workflow_hyde.add_edge("hyde_retriever", "generator")
workflow_hyde.add_edge("generator", END)

app_hyde = workflow_hyde.compile()

In [13]:
from typing import TypedDict, Annotated
import operator

class DecompState(TypedDict):
    messages: Annotated[list, operator.add]
    sub_questions: list[str]
    sub_contexts: list[str]


def decompose_node(state: DecompState):
    """
    Разбивает сложный вопрос на подвопросы.
    """
    user_question = state["messages"][-1].content

    decompose_prompt = (
        "Разбей следующий вопрос на 2-3 простых подвопроса для поиска в базе знаний. "
        "Верни только список подвопросов, каждый с новой строки, без нумерации и пояснений.\n\n"
        f"Вопрос: {user_question}"
    )

    result = llm.invoke([HumanMessage(content=decompose_prompt)])
    sub_questions = [q.strip() for q in result.content.strip().split('\n') if q.strip()]

    return {"sub_questions": sub_questions}


def retrieve_all_node(state: DecompState):
    """
    Делает поиск по каждому подвопросу отдельно.
    """
    sub_questions = state.get("sub_questions", [])
    sub_contexts = []

    for question in sub_questions:
        try:
            dense_vec = bge.encode(question)['dense_vecs']
            results = qdrant_client.query_points(
                collection_name=collection_name,
                query=dense_vec,
                limit=2
            )
            if results.points:
                chunk = results.points[0].payload.get('page_content', '')
                sub_contexts.append(f"[Подвопрос: {question}]\n{chunk}")
            else:
                sub_contexts.append(f"[Подвопрос: {question}]\nКонтекст не найден.")
        except Exception as e:
            print(f"Ошибка поиска для подвопроса '{question}': {e}")
            sub_contexts.append(f"[Подвопрос: {question}]\nОшибка поиска.")

    return {"sub_contexts": sub_contexts}


def merge_context_node(state: DecompState):
    """
    Объединяет найденные контексты по всем подвопросам.
    """
    sub_contexts = state.get("sub_contexts", [])
    merged = "\n\n".join(sub_contexts) if sub_contexts else "Контекст отсутствует."
    return {
        "messages": [AIMessage(content=merged, name="context_holder")]
    }


workflow_decomp = StateGraph(DecompState)
workflow_decomp.add_node("decompose", decompose_node)
workflow_decomp.add_node("retrieve_all", retrieve_all_node)
workflow_decomp.add_node("merge_context", merge_context_node)
workflow_decomp.add_node("generator", generate_node)

workflow_decomp.add_edge(START, "decompose")
workflow_decomp.add_edge("decompose", "retrieve_all")
workflow_decomp.add_edge("retrieve_all", "merge_context")
workflow_decomp.add_edge("merge_context", "generator")
workflow_decomp.add_edge("generator", END)

app_decomp = workflow_decomp.compile()

# СРАВНЕНИЕ: Baseline, HyDE, Decomposition

In [14]:
import pandas as pd

test_queries = [
    "Каков порядок возврата товара?",
    "Какие документы нужны для оформления гарантийного ремонта?",
    "Как оформить корпоративный заказ?",
    "Что делать если покупатель хочет обменять телефон?",
]

def run_pipeline(app, query: str, state_class=None) -> str:
    """Запускает пайплайн и возвращает финальный ответ."""
    if state_class == "decomp":
        init_state = {
            "messages": [HumanMessage(content=query)],
            "sub_questions": [],
            "sub_contexts": []
        }
    else:
        init_state = {"messages": [HumanMessage(content=query)]}

    result = app.invoke(init_state)
    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage) and not hasattr(msg, 'name'):
            return msg.content
        if isinstance(msg, AIMessage) and msg.name is None:
            return msg.content
    return result["messages"][-1].content


results = []
for query in test_queries:
    print(f"\nЗапрос: {query}")

    baseline_ans = run_pipeline(app, query)
    print(f"[Baseline] {baseline_ans[:250]}...")

    hyde_ans = run_pipeline(app_hyde, query)
    print(f"[HyDE]     {hyde_ans[:250]}...")

    decomp_ans = run_pipeline(app_decomp, query, state_class="decomp")
    print(f"[Decomp]   {decomp_ans[:250]}...")

    results.append({
        "Запрос": query,
        "Baseline": baseline_ans,
        "HyDE": hyde_ans,
        "Декомпозиция": decomp_ans
    })

df_results = pd.DataFrame(results)
df_results[["Запрос", "Baseline", "HyDE", "Декомпозиция"]].head()


Запрос: Каков порядок возврата товара?
[Baseline] Для осуществления возврата товара абонент обязан предъявить заключение сервисного центра Принципала о том, что данный конкретный товар соответствует условиям обмена или возврата товара. Возврат денежных средств юридическому лицу при оплате покупки бе...
[HyDE]     Возврат денежных средств физическому лицу производится наличным способом через кассу в течение 10 (десяти) рабочих дней после обращения....
[Decomp]   В течение четырнадцати (14) календарных дней с момента покупки товара; 
При наличии документов, подтверждающих факт и дату покупки (гарантийный талон, копия товарного или кассового чека, счет фактуры); 
При условии, что товар исправен, сохранен товар...

Запрос: Какие документы нужны для оформления гарантийного ремонта?
[Baseline] Для оформления гарантийного ремонта необходимы следующие документы: 
1. Правильно и без помарок и исправлений заполненный гарантийный талон, в котором должны быть указаны модель и серийный номер издел

,Запрос,Baseline,HyDE,Декомпозиция
0,Каков порядок возврата товара?,Для осуществления возврата товара абонент обяз...,Возврат денежных средств физическому лицу прои...,В течение четырнадцати (14) календарных дней с...
1,Какие документы нужны для оформления гарантийн...,Для оформления гарантийного ремонта необходимы...,Для оформления гарантийного ремонта необходимы...,Для оформления гарантийного ремонта необходимы...
2,Как оформить корпоративный заказ?,"К сожалению, я не могу ответить на этот вопрос...","К сожалению, я не могу ответить на этот вопрос...",Для оформления корпоративного заказа необходим...
3,Что делать если покупатель хочет обменять теле...,"Если покупатель хочет обменять телефон, он дол...","Если покупатель хочет обменять телефон, ему бу...","Если покупатель хочет обменять телефон, необхо..."
